In [1]:
import tensorflow as tf
import numpy as np
import time

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l1, l2


# ============================================================
# 1. LOAD CIFAR-10 DATASET
# ============================================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Training data:", x_train.shape)
print("Testing data :", x_test.shape)


# ============================================================
# 2. PREPROCESSING
# ============================================================

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)


# ============================================================
# 3. SETTINGS
# ============================================================

EPOCHS = 5
BATCH_SIZE = 64


# ============================================================
# 4. FUNCTION TO CREATE MODEL
# ============================================================
def create_model(model_type):

    # ---------------- BASELINE ----------------
    if model_type == "Baseline":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(512, activation="relu"),
            Dense(256, activation="relu"),
            Dense(64, activation="relu"),

            Dense(10, activation="softmax")
        ])

        regularization = "None"


    # ---------------- XAVIER ----------------
    elif model_type == "Xavier":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(
                512,
                activation="relu",
                kernel_initializer="glorot_uniform"
            ),

            Dense(
                256,
                activation="relu",
                kernel_initializer="glorot_uniform"
            ),

            Dense(
                64,
                activation="relu",
                kernel_initializer="glorot_uniform"
            ),

            Dense(10, activation="softmax")
        ])

        regularization = "None"


    # ---------------- KAIMING / HE ----------------
    elif model_type == "Kaiming/He":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(
                512,
                activation="relu",
                kernel_initializer="he_normal"
            ),

            Dense(
                256,
                activation="relu",
                kernel_initializer="he_normal"
            ),

            Dense(
                64,
                activation="relu",
                kernel_initializer="he_normal"
            ),

            Dense(10, activation="softmax")
        ])

        regularization = "None"


    # ---------------- DROPOUT ----------------
    elif model_type == "Dropout":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(512, activation="relu"),
            Dropout(0.5),

            Dense(256, activation="relu"),
            Dropout(0.5),

            Dense(64, activation="relu"),
            Dropout(0.5),

            Dense(10, activation="softmax")
        ])

        regularization = "Dropout (0.5)"


    # ---------------- L1 ----------------
    elif model_type == "L1":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(
                512,
                activation="relu",
                kernel_regularizer=l1(0.001)
            ),

            Dense(
                256,
                activation="relu",
                kernel_regularizer=l1(0.001)
            ),

            Dense(
                64,
                activation="relu",
                kernel_regularizer=l1(0.001)
            ),

            Dense(10, activation="softmax")
        ])

        regularization = "L1 (0.001)"


    # ---------------- L2 ----------------
    elif model_type == "L2":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(
                512,
                activation="relu",
                kernel_regularizer=l2(0.001)
            ),

            Dense(
                256,
                activation="relu",
                kernel_regularizer=l2(0.001)
            ),

            Dense(
                64,
                activation="relu",
                kernel_regularizer=l2(0.001)
            ),

            Dense(10, activation="softmax")
        ])

        regularization = "L2 (0.001)"


    return model, regularization
# ============================================================
# 5. MODELS AND OPTIMIZERS
# ============================================================
model_types = [
    "Baseline",
    "Xavier",
    "Kaiming/He",
    "Dropout",
    "L1",
    "L2"
]

optimizers = {
    "SGD": tf.keras.optimizers.SGD(learning_rate=0.01),
    "Adam": tf.keras.optimizers.Adam(learning_rate=0.001)
}


# ============================================================
# 6. STORE RESULTS
# ============================================================

results = []


# ============================================================
# 7. TRAIN ALL MODELS WITH SGD AND ADAM
# ============================================================

for model_type in model_types:

    for optimizer_name in optimizers:

        print("\n")
        print("=" * 60)
        print("Model     :", model_type)
        print("Optimizer :", optimizer_name)
        print("=" * 60)

        # Create a fresh model
        model, regularization = create_model(model_type)

        # Create a fresh optimizer
        if optimizer_name == "SGD":
            optimizer = tf.keras.optimizers.SGD(
                learning_rate=0.01
            )
        else:
            optimizer = tf.keras.optimizers.Adam(
                learning_rate=0.001
            )

        # Compile
        model.compile(
            optimizer=optimizer,
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

        # Start timer
        start_time = time.time()

        # Train
        model.fit(
            x_train,
            y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=0.2,
            verbose=1
        )

        # Training time
        training_time = time.time() - start_time

        # Evaluate
        test_loss, test_accuracy = model.evaluate(
            x_test,
            y_test,
            verbose=0
        )

        # Print result
        print("\nTest Accuracy:",
              round(test_accuracy * 100, 2), "%")

        print("Test Loss:",
              round(test_loss, 4))

        print("Training Time:",
              round(training_time, 2), "seconds")

        # Store results
        results.append([
            model_type,
            regularization,
            optimizer_name,
            test_accuracy * 100,
            test_loss,
            training_time
        ])


# ============================================================
# 8. DISPLAY FINAL RESULTS
# ============================================================

print("\n\n")
print("=" * 105)
print("                         FINAL RESULTS")
print("=" * 105)

print(
    f"{'Model':<18}"
    f"{'Regularization':<20}"
    f"{'Optimizer':<12}"
    f"{'Test Accuracy (%)':<20}"
    f"{'Test Loss':<12}"
    f"{'Training Time (s)':<20}"
)

print("-" * 105)


for result in results:

    model_name = result[0]
    regularization = result[1]
    optimizer = result[2]
    accuracy = result[3]
    loss = result[4]
    training_time = result[5]

    print(
        f"{model_name:<18}"
        f"{regularization:<20}"
        f"{optimizer:<12}"
        f"{accuracy:<20.2f}"
        f"{loss:<12.4f}"
        f"{training_time:<20.2f}"
    )


print("=" * 105)


# ============================================================
# 9. FIND BEST RESULT
# ============================================================

best_result = max(results, key=lambda x: x[3])

print("\n")
print("==========================================")
print("              BEST RESULT")
print("==========================================")

print("Model           :", best_result[0])
print("Regularization  :", best_result[1])
print("Optimizer       :", best_result[2])
print("Test Accuracy   :", round(best_result[3], 2), "%")
print("Test Loss       :", round(best_result[4], 4))
print("Training Time   :", round(best_result[5], 2), "seconds")

C:\Users\Student\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Training data: (50000, 32, 32, 3)
Testing data : (10000, 32, 32, 3)


Model     : Baseline
Optimizer : SGD
Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.2995 - loss: 1.9577 - val_accuracy: 0.3487 - val_loss: 1.8445
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.3711 - loss: 1.7702 - val_accuracy: 0.3582 - val_loss: 1.8024
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.4017 - loss: 1.6866 - val_accuracy: 0.4083 - val_loss: 1.6710
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.4214 - loss: 1.6322 - val_accuracy: 0.4247 - val_loss: 1.6288
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.4346 - loss: 1.5908 - val_accuracy: 0.4339 - val_loss: 1.6170

Test Accuracy: 44.11 %
Test Loss: 1.5847
Training Time: 18.98 seconds


Model     : Baseline
Optimizer : Adam
Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.3107 - loss: 1.8996 - val_accuracy: 0.3703 - val_loss: 1.7459
Epoch 2/5
625/625 ━

In [2]:
import tensorflow as tf
import numpy as np
import time

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l1, l2


# ============================================================
# 1. LOAD CIFAR-10 DATASET
# ============================================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Training data:", x_train.shape)
print("Testing data :", x_test.shape)


# ============================================================
# 2. PREPROCESSING
# ============================================================

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)


# ============================================================
# 3. SETTINGS
# ============================================================

EPOCHS = 5
BATCH_SIZE = 64


# ============================================================
# 4. FUNCTION TO CREATE MODEL
# ============================================================
def create_model(model_type):

    # ---------------- BASELINE ----------------
    if model_type == "Baseline":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(512, activation="relu"),
            Dense(256, activation="relu"),
            Dense(64, activation="relu"),

            Dense(10, activation="softmax")
        ])

        regularization = "None"


    # ---------------- XAVIER ----------------
    elif model_type == "Xavier":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(
                512,
                activation="relu",
                kernel_initializer="glorot_uniform"
            ),

            Dense(
                256,
                activation="relu",
                kernel_initializer="glorot_uniform"
            ),

            Dense(
                64,
                activation="relu",
                kernel_initializer="glorot_uniform"
            ),

            Dense(10, activation="softmax")
        ])

        regularization = "None"


    # ---------------- KAIMING / HE ----------------
    elif model_type == "Kaiming/He":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(
                512,
                activation="relu",
                kernel_initializer="he_normal"
            ),

            Dense(
                256,
                activation="relu",
                kernel_initializer="he_normal"
            ),

            Dense(
                64,
                activation="relu",
                kernel_initializer="he_normal"
            ),

            Dense(10, activation="softmax")
        ])

        regularization = "None"


    # ---------------- DROPOUT ----------------
    elif model_type == "Dropout":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(512, activation="relu"),
            Dropout(0.5),

            Dense(256, activation="relu"),
            Dropout(0.5),

            Dense(64, activation="relu"),
            Dropout(0.5),

            Dense(10, activation="softmax")
        ])

        regularization = "Dropout (0.5)"


    # ---------------- L1 ----------------
    elif model_type == "L1":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(
                512,
                activation="relu",
                kernel_regularizer=l1(0.001)
            ),

            Dense(
                256,
                activation="relu",
                kernel_regularizer=l1(0.001)
            ),

            Dense(
                64,
                activation="relu",
                kernel_regularizer=l1(0.001)
            ),

            Dense(10, activation="softmax")
        ])

        regularization = "L1 (0.001)"


    # ---------------- L2 ----------------
    elif model_type == "L2":

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten(),

            Dense(
                512,
                activation="relu",
                kernel_regularizer=l2(0.001)
            ),

            Dense(
                256,
                activation="relu",
                kernel_regularizer=l2(0.001)
            ),

            Dense(
                64,
                activation="relu",
                kernel_regularizer=l2(0.001)
            ),

            Dense(10, activation="softmax")
        ])

        regularization = "L2 (0.001)"


    return model, regularization
# ============================================================
# 5. MODELS AND OPTIMIZERS
# ============================================================
model_types = [
    "Baseline",
    "Xavier",
    "Kaiming/He",
    "Dropout",
    "L1",
    "L2"
]

optimizers = {
    "SGD": tf.keras.optimizers.SGD(learning_rate=0.01),
    "Adam": tf.keras.optimizers.Adam(learning_rate=0.001)
}


# ============================================================
# 6. STORE RESULTS
# ============================================================

results = []


# ============================================================
# 7. TRAIN ALL MODELS WITH SGD AND ADAM
# ============================================================

for model_type in model_types:

    for optimizer_name in optimizers:

        print("\n")
        print("=" * 60)
        print("Model     :", model_type)
        print("Optimizer :", optimizer_name)
        print("=" * 60)

        # Create a fresh model
        model, regularization = create_model(model_type)

        # Create a fresh optimizer
        if optimizer_name == "SGD":
            optimizer = tf.keras.optimizers.SGD(
                learning_rate=0.01
            )
        else:
            optimizer = tf.keras.optimizers.Adam(
                learning_rate=0.001
            )

        # Compile
        model.compile(
            optimizer=optimizer,
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

        # Start timer
        start_time = time.time()

        # Train
        model.fit(
            x_train,
            y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=0.2,
            verbose=1
        )

        # Training time
        training_time = time.time() - start_time

        # Evaluate
        test_loss, test_accuracy = model.evaluate(
            x_test,
            y_test,
            verbose=0
        )

        # Print result
        print("\nTest Accuracy:",
              round(test_accuracy * 100, 2), "%")

        print("Test Loss:",
              round(test_loss, 4))

        print("Training Time:",
              round(training_time, 2), "seconds")

        # Store results
        results.append([
            model_type,
            regularization,
            optimizer_name,
            test_accuracy * 100,
            test_loss,
            training_time
        ])


# ============================================================
# 8. DISPLAY FINAL RESULTS
# ============================================================

print("\n\n")
print("=" * 105)
print("                         FINAL RESULTS")
print("=" * 105)

print(
    f"{'Model':<18}"
    f"{'Regularization':<20}"
    f"{'Optimizer':<12}"
    f"{'Test Accuracy (%)':<20}"
    f"{'Test Loss':<12}"
    f"{'Training Time (s)':<20}"
)

print("-" * 105)


for result in results:

    model_name = result[0]
    regularization = result[1]
    optimizer = result[2]
    accuracy = result[3]
    loss = result[4]
    training_time = result[5]

    print(
        f"{model_name:<18}"
        f"{regularization:<20}"
        f"{optimizer:<12}"
        f"{accuracy:<20.2f}"
        f"{loss:<12.4f}"
        f"{training_time:<20.2f}"
    )


print("=" * 105)


# ============================================================
# 9. FIND BEST RESULT
# ============================================================

best_result = max(results, key=lambda x: x[3])

print("\n")
print("==========================================")
print("              BEST RESULT")
print("==========================================")

print("Model           :", best_result[0])
print("Regularization  :", best_result[1])
print("Optimizer       :", best_result[2])
print("Test Accuracy   :", round(best_result[3], 2), "%")
print("Test Loss       :", round(best_result[4], 4))
print("Training Time   :", round(best_result[5], 2), "seconds")

Training data: (50000, 32, 32, 3)
Testing data : (10000, 32, 32, 3)


Model     : Baseline
Optimizer : SGD
Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.2946 - loss: 1.9644 - val_accuracy: 0.3355 - val_loss: 1.8435
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.3701 - loss: 1.7730 - val_accuracy: 0.3758 - val_loss: 1.7485
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.3985 - loss: 1.6897 - val_accuracy: 0.4076 - val_loss: 1.6694
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.4209 - loss: 1.6337 - val_accuracy: 0.4362 - val_loss: 1.6173
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.4390 - loss: 1.5861 - val_accuracy: 0.4317 - val_loss: 1.6033

Test Accuracy: 44.01 %
Test Loss: 1.5705
Training Time: 19.74 seconds


Model     : Baseline
Optimizer : Adam
Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.3059 - loss: 1.9084 - val_accuracy: 0.3681 - val_loss: 1.7473
Epoch 2/5
625/625 ━

625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.1657 - loss: 2.1598 - val_accuracy: 0.1909 - val_loss: 2.0949

Test Accuracy: 19.19 %
Test Loss: 2.094
Training Time: 41.4 seconds


Model     : L1
Optimizer : SGD
Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.3002 - loss: 35.9645 - val_accuracy: 0.3529 - val_loss: 31.0094
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.3601 - loss: 26.6759 - val_accuracy: 0.3767 - val_loss: 22.5925
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.3733 - loss: 19.0774 - val_accuracy: 0.3713 - val_loss: 15.8300
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.3712 - loss: 13.1026 - val_accuracy: 0.3508 - val_loss: 10.6742
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.3614 - loss: 8.7446 - val_accuracy: 0.3211 - val_loss: 7.1991

Test Accuracy: 32.74 %
Test Loss: 7.1839
Training Time: 28.77 seconds


Model     : L1
Optimizer : Adam
Epoch 1/5
625/625 ━━━━━━━━━━━━━